# NLP Lecture Notebook: From Transformers to Modern LLMs

Welcome to the practical companion for our lecture on modern NLP. In this notebook, we will implement the core concepts discussed, from tokenization to interacting with local and cloud-based Large Language Models.

# Homework

Everything below runs **locally against Ollama** — no OpenAI API key required. Ollama exposes an OpenAI-compatible endpoint, so we reuse the exact same SDK and the agent loop from section 3.3.

**Before you start** (once, from a terminal):
```
ollama serve
ollama pull qwen2.5
```
Run the setup cell, then fill in the `# TODO`s in each task.

In [ ]:
# --- Homework setup: local model via Ollama, no OpenAI key needed ---
import json
from openai import OpenAI

# Ollama speaks the OpenAI API on localhost:11434/v1 -> reuse the same client.
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
OLLAMA_MODEL = "qwen2.5"          # must support tool calling

# Smoke test (should print something like "ready"):
r = ollama_client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": "Reply with one word: ready"}])
print("Model says:", r.choices[0].message.content)

# The agent loop from section 3.3, parameterised for any client/model:
def run_agent_local(user_prompt, tools, dispatch,
                    client=ollama_client, model=OLLAMA_MODEL, max_steps=5):
    messages = [{"role": "user", "content": user_prompt}]
    for step in range(max_steps):
        resp = client.chat.completions.create(
            model=model, messages=messages, tools=tools, tool_choice="auto")
        msg = resp.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"  [step {step}] -> {tc.function.name}({args})")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(dispatch[tc.function.name](**args))})
    return "Stopped: hit max_steps."

Model says: Ready


**▶️ Expected output** (this cell runs as-is)

```
Model says: ready
```

If you get a connection error, Ollama isn't running (`ollama serve`); if the model isn't found, run `ollama pull llama3.1` first.

### Task 1 — Add a third tool and watch the model chain

Give the agent a **currency converter** on top of the price and calculator tools, then ask a question that forces all three (get price → multiply by quantity → convert). Fill in the four TODOs and run it.

In [ ]:
# Tools the agent already knows (from section 3.3):
def get_stock_price(ticker: str):
    prices = {"NVDA": 125.50, "GOOG": 178.20, "AAPL": 229.00}
    return {"ticker": ticker.upper(), "price_usd": prices.get(ticker.upper(), 0.0)}

def calculator(expression: str):
    return {"expression": expression, "result": eval(expression)}  # demo only

# TODO 1: finish the third tool — convert a USD amount to another currency.
def convert_currency(amount_usd: float, to: str):
    rates = {"EUR": 0.92, "GBP": 0.79, "UAH": 41.0}
    return {"amount": float(amount_usd) * rates[str(to).upper()], "currency": to}

TOOLS = [
    {"type": "function", "function": {
        "name": "get_stock_price",
        "description": "Get the latest share price (USD) for a ticker symbol",
        "parameters": {"type": "object",
            "properties": {"ticker": {"type": "string"}}, "required": ["ticker"]}}},
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate an arithmetic expression like '125.5 * 10'",
        "parameters": {"type": "object",
            "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
    # TODO 2: add the function spec for convert_currency here
   {"type": "function", "function": {
        "name": "convert_currency",
        "description": "Convert a USD amount to another currency (EUR, GBP, or UAH)",
        "parameters": {"type": "object",
            "properties": {"amount_usd": {"type": "number", "description": "The amount in USD to convert"},
                "to": {"type": "string", "description": "Target currency code, e.g. 'EUR', 'GBP', 'UAH'"}
        },"required": ["amount_usd", "to"]}}},
]

# TODO 3: register the new tool so the loop can dispatch to it
DISPATCH = {"get_stock_price": get_stock_price, "calculator": calculator, "convert_currency": convert_currency}

# TODO 4: ask something that needs all three tools, then run it:
print(run_agent_local("How much is 10 shares of NVDA worth in euros? For calculation use provided tools only", TOOLS, DISPATCH))

  [step 0] -> get_stock_price({'ticker': 'NVDA'})
  [step 1] -> calculator({'expression': '125.5 * 10'})
  [step 2] -> convert_currency({'amount_usd': 1255, 'to': 'EUR'})
The value of 10 shares of NVDA is currently approximately 1154.60 euros.


**▶️ Expected output** (after completing TODO 1–4 and uncommenting the last line)

```
  [step 0] -> get_stock_price({'ticker': 'NVDA'})
  [step 1] -> calculator({'expression': '125.5 * 10'})
  [step 2] -> convert_currency({'amount_usd': 1255.0, 'to': 'EUR'})
10 shares of NVDA are worth about €1,154.60 (1255 USD × 0.92).
```

The model chained all three tools on its own. Exact wording, the order of the middle steps, and rounding will vary between runs and models — what matters is that **all three tools get called** and the final number is right.

### Task 2 — Structured output, locally

Section 3.4 used OpenAI's schema-guaranteed parsing. Ollama does the same trick with a JSON schema. Define a Pydantic model for a product review and extract it from the paragraph below — the reply is guaranteed to match your schema.

In [ ]:
import ollama
from pydantic import BaseModel

# TODO 1: define the schema — sentiment (str), score (int, 1-5),
#         pros (list[str]), cons (list[str])
class Review(BaseModel):
    sentiment: str
    score: int
    cons: list[str]
    pros: list[str]

text = ("Battery life is fantastic and it's super light, but the camera is "
        "mediocre in low light and the price is a bit high.")

# TODO 2: call ollama with format=<your schema>, then validate the JSON reply.
resp = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": f"Extract a product review as JSON:\n{text}"}],
    format=Review.model_json_schema())
review = Review.model_validate_json(resp.message.content)
print(review)

sentiment='Mixed' score=3 cons=['Camera performance is poor in low light'] pros=['Battery life is excellent', 'Device is lightweight']


**▶️ Expected output** (after completing TODO 1–2)

```
sentiment='mixed' score=3 pros=['fantastic battery life', 'lightweight'] cons=['mediocre low-light camera', 'price a bit high']
```

`review` is a real, validated `Review` object — the JSON came back matching your schema exactly. The phrasing of the pros/cons will differ per run; the structure won't.

### Task 3 — Extend the MCP server (no model needed)

Add a second tool to the weather server, then use the MCP client from section 5.2 to confirm **both** tools are discovered and call the new one. This exercises the protocol directly — no LLM involved. We write to a separate file (`weather_server_hw.py`) so the section-5 demo server stays intact.

In [ ]:
%%writefile weather_server_hw.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather-hw")

@mcp.tool()
def get_forecast(city: str) -> str:
    """Return a short weather forecast for a city."""
    data = {"Kyiv": "18C, partly cloudy", "London": "12C, rain", "Tokyo": "24C, clear"}
    return data.get(city, f"No forecast available for {city}")

# TODO 1: add a get_air_quality(city) tool that returns an AQI string per city.
@mcp.tool()
def get_air_quality(city: str) -> str:
    """Return the air-quality index (AQI) for a city."""
    data = {"Kyiv": "AQI 42 (good)", "London": "AQI 68 (moderate)", "Tokyo": "AQI 35 (good)"}
    return data.get(city, "unknown")

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting weather_server_hw.py


**▶️ Expected output**

```
Writing weather_server_hw.py
```

(`Overwriting weather_server_hw.py` if you run it again.) The cell just saves the file — it does **not** start the server here.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import subprocess

server_hw = StdioServerParameters(command="python", args=["weather_server_hw.py"])

async def check():
    async with stdio_client(server_hw, errlog=subprocess.DEVNULL) as (read, write): # fix for Windows problem with sys.stderr and fileno()
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("Discovered tools:", [t.name for t in tools.tools])  # expect BOTH now
            # TODO 2: call get_air_quality for a city and print result.content[0].text
            result = await session.call_tool("get_air_quality", {"city": "Kyiv"})
            print("Kyiv air quality:", result.content[0].text)

await check()   # in a .py script: asyncio.run(check())

Discovered tools: ['get_forecast', 'get_air_quality']
Kyiv air quality: AQI 42 (good)


**▶️ Expected output** (after completing TODO 2)

```
Discovered tools: ['get_forecast', 'get_air_quality']
Kyiv air quality: AQI 42 (good)
```

Both tools are now visible over the protocol, and the new one returns your data — all with no LLM involved.

### Task 4 — Reading (no code)

- Skim the MCP intro: https://modelcontextprotocol.io
- Hugging Face text-classification fine-tuning guide: https://huggingface.co/docs/transformers/tasks/sequence_classification

**Stretch (optional):** combine Task 1 and Task 3 — point `run_agent_local` at the MCP tools (adapt `run_mcp_agent` from section 5.3 to use `ollama_client` instead of `client`) so a *local* model answers "Is the air in Kyiv safe today?" by calling your MCP server.